In [1]:
# ============================================================
# AMPds2 — RAW (no-tsfresh) Feature Pipeline
# Sept-Oct 2012 subset (2 months) | Synthetic anomaly injection |
# Simple per-window aggregate stats instead of tsfresh | Context vector
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATA_DIR = r'C:\1.Revanth\Projects\research'
DATE_START = '2012-09-01'
DATE_END   = '2012-10-31'
WINDOW_SIZE_MIN = 15
APPLIANCE_COLS  = ['FRE', 'HPE', 'DWE', 'CWE', 'WOE', 'B1E']

CACHE_DIR = os.path.join(DATA_DIR, 'pipeline_cache_raw')   # separate from tsfresh cache
os.makedirs(CACHE_DIR, exist_ok=True)

np.random.seed(42)

# ============================================================
# 1. LOAD DATA
# ============================================================
whe     = pd.read_csv(os.path.join(DATA_DIR, 'Electricity_WHE.csv'))
p_all   = pd.read_csv(os.path.join(DATA_DIR, 'Electricity_P.csv'))
weather = pd.read_csv(os.path.join(DATA_DIR, 'Climate_HourlyWeather.csv'))

# ============================================================
# 2. TIMESTAMP ALIGNMENT
# ============================================================
whe_ts_col = 'unix_ts' if 'unix_ts' in whe.columns else 'UNIX_TS'
p_ts_col   = 'UNIX_TS' if 'UNIX_TS' in p_all.columns else 'unix_ts'

whe['timestamp']   = pd.to_datetime(whe[whe_ts_col], unit='s', utc=True).dt.tz_convert('America/Vancouver')
p_all['timestamp'] = pd.to_datetime(p_all[p_ts_col], unit='s', utc=True).dt.tz_convert('America/Vancouver')

whe   = whe.drop(columns=[whe_ts_col]).set_index('timestamp').sort_index()
p_all = p_all.drop(columns=[p_ts_col]).set_index('timestamp').sort_index()

weather['timestamp'] = pd.to_datetime(weather['Date/Time'], format='mixed')
weather = weather.set_index('timestamp').sort_index()
weather.index = weather.index.tz_localize('America/Vancouver', ambiguous='NaT', nonexistent='shift_forward')
weather = weather[weather.index.notna()]

weather = weather.drop(columns=[
    'Data Quality', 'Temp Flag', 'Dew Point Temp Flag', 'Rel Hum Flag',
    'Wind Dir Flag', 'Wind Spd Flag', 'Visibility Flag', 'Stn Press Flag',
    'Hmdx', 'Hmdx Flag', 'Wind Chill', 'Wind Chill Flag'
], errors='ignore')

# ============================================================
# 3. FILTER TO 2-MONTH SUBSET
# ============================================================
whe     = whe.loc[DATE_START:DATE_END]
p_all   = p_all.loc[DATE_START:DATE_END]
weather = weather.loc[DATE_START:DATE_END]

print(f"WHE rows: {len(whe)}, P rows: {len(p_all)}, weather rows: {len(weather)}")

# ============================================================
# 4. MERGE WHE (mains) + selected appliances from P.csv
# ============================================================
appliance_cols_present = [c for c in APPLIANCE_COLS if c in p_all.columns]
merged = whe.join(p_all[appliance_cols_present], how='inner')

BEHAVIOR_COLS = ['V', 'I', 'P', 'Q', 'S'] + appliance_cols_present

for col in BEHAVIOR_COLS:
    merged[col] = pd.to_numeric(merged[col], errors='coerce').astype('float64')
merged[BEHAVIOR_COLS] = merged[BEHAVIOR_COLS].ffill().bfill()

# ============================================================
# 5. WEATHER: encode regime + merge into minute-level data
# ============================================================
def simplify_weather(w):
    if pd.isna(w):
        return 'Unknown'
    w = w.lower()
    if 'thunder' in w:
        return 'Thunderstorm'
    if 'snow' in w:
        return 'Snow'
    if 'fog' in w:
        return 'Fog'
    if 'rain' in w or 'drizzle' in w:
        return 'Rain'
    if 'cloud' in w:
        return 'Cloudy'
    if 'clear' in w or 'sunny' in w:
        return 'Clear'
    return 'Other'

weather['weather_regime'] = weather['Weather'].apply(simplify_weather)
weather_dummies = pd.get_dummies(weather['weather_regime'], prefix='wx', dtype=int)
weather = weather.join(weather_dummies)

weather_numeric_cols = ['Temp (C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']
for col in weather_numeric_cols:
    weather[col] = pd.to_numeric(weather[col], errors='coerce')

weather_regime_cols = weather_dummies.columns.tolist()

merged = merged.join(weather[weather_numeric_cols + weather_regime_cols], how='left')
merged[weather_numeric_cols + weather_regime_cols] = merged[weather_numeric_cols + weather_regime_cols].ffill().infer_objects(copy=False)

# ============================================================
# 6. SYNTHETIC ANOMALY INJECTION — same as tsfresh version
# ============================================================
merged['window_id'] = merged.index.floor(f'{WINDOW_SIZE_MIN}min')
anomaly_log = []

def inject_heating_on_warm_day(df, log, n_events=65):
    candidates = df[(df['wx_Clear'] == 1) & (df['Temp (C)'] > 15)].index
    if len(candidates) == 0:
        return df
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'HPE'] = df.loc[window, 'HPE'].max() * np.random.uniform(3, 5) + 500
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'heating_on_warm_day'})
    return df

def inject_appliance_unusual_hours(df, log, appliance='WOE', n_events=80):
    candidates = df[(df.index.hour >= 2) & (df.index.hour <= 5)].index
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    base = df[appliance].quantile(0.9)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, appliance] = base * np.random.uniform(1.5, 3)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'appliance_unusual_hours'})
    return df

def inject_high_usage_low_occupancy(df, log, n_events=80):
    candidates = df[(df.index.hour >= 10) & (df.index.hour <= 15) & (df.index.dayofweek < 5)].index
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'P'] = df.loc[window, 'P'] + np.random.uniform(700, 1400)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'high_usage_low_occupancy'})
    return df

def inject_weekend_pattern_on_weekday(df, log, n_events=65):
    weekday_candidates = df[df.index.dayofweek < 5].index
    weekend_avg = df[df.index.dayofweek >= 5]['CWE'].mean()
    chosen = np.random.choice(weekday_candidates, size=min(n_events, len(weekday_candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'CWE'] = weekend_avg * np.random.uniform(1.5, 2.5)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'weekend_pattern_on_weekday'})
    return df

def inject_weekday_pattern_on_weekend(df, log, n_events=65):
    weekend_candidates = df[df.index.dayofweek >= 5].index
    weekday_avg = df[df.index.dayofweek < 5]['CWE'].mean()
    chosen = np.random.choice(weekend_candidates, size=min(n_events, len(weekend_candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'CWE'] = weekday_avg * np.random.uniform(1.5, 2.5)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'weekday_pattern_on_weekend'})
    return df

def inject_stuck_appliance_on(df, log, appliance='DWE', n_events=16, duration_min=90):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    stuck_value = df[appliance].quantile(0.75)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        df.loc[window, appliance] = stuck_value
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'stuck_appliance_on'})
    return df

def inject_stuck_appliance_off(df, log, appliance='HPE', n_events=16, duration_min=90):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        df.loc[window, appliance] = 0.0
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'stuck_appliance_off'})
    return df

def inject_gradual_drift(df, log, appliance='B1E', n_events=20, duration_min=60):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        direction = np.random.choice([1, -1])
        ramp = np.linspace(1.0, 1.0 + direction * np.random.uniform(0.8, 1.5), len(window))
        df.loc[window, appliance] = df.loc[window, appliance] * ramp
        label = 'gradual_drift_increase' if direction == 1 else 'gradual_drift_decrease'
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': label})
    return df

def inject_sustained_overload(df, log, n_events=16, duration_min=60):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        df.loc[window, 'P'] = df.loc[window, 'P'] + np.random.uniform(1000, 1800)
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'sustained_overload'})
    return df

def inject_multiple_high_power_simultaneous(df, log, appliances=('HPE', 'DWE', 'CWE'), n_events=40):
    chosen = np.random.choice(df.index, size=min(n_events, len(df.index)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        for app in appliances:
            df.loc[window, app] = df[app].quantile(0.9) * np.random.uniform(1.2, 1.8)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'multiple_high_power_simultaneous'})
    return df

def inject_impossible_appliance_combo(df, log, appliances=('HPE', 'WOE'), n_events=33):
    chosen = np.random.choice(df.index, size=min(n_events, len(df.index)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        for app in appliances:
            df.loc[window, app] = df[app].quantile(0.95) * np.random.uniform(1.3, 2.0)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'impossible_appliance_combo'})
    return df

def inject_power_spike(df, log, n_events=40):
    chosen = np.random.choice(df.index, size=min(n_events, len(df.index)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'P'] = df.loc[window, 'P'] + np.random.uniform(900, 1700)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'power_spike'})
    return df

def inject_sensor_glitch(df, log, n_events=26):
    chosen = np.random.choice(df.index, size=min(n_events, len(df.index)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'V'] = df.loc[window, 'V'] * np.random.uniform(0.5, 1.8)
        df.loc[window, 'I'] = df.loc[window, 'I'] * np.random.uniform(0.3, 2.5)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'sensor_glitch'})
    return df

merged = inject_heating_on_warm_day(merged, anomaly_log)
merged = inject_appliance_unusual_hours(merged, anomaly_log)
merged = inject_high_usage_low_occupancy(merged, anomaly_log)
merged = inject_weekend_pattern_on_weekday(merged, anomaly_log)
merged = inject_weekday_pattern_on_weekend(merged, anomaly_log)
merged = inject_stuck_appliance_on(merged, anomaly_log)
merged = inject_stuck_appliance_off(merged, anomaly_log)
merged = inject_gradual_drift(merged, anomaly_log)
merged = inject_sustained_overload(merged, anomaly_log)
merged = inject_multiple_high_power_simultaneous(merged, anomaly_log)
merged = inject_impossible_appliance_combo(merged, anomaly_log)
merged = inject_power_spike(merged, anomaly_log)
merged = inject_sensor_glitch(merged, anomaly_log)

anomaly_df = pd.DataFrame(anomaly_log).drop_duplicates(subset='window_id')
print("\nInjected anomaly counts by type:")
print(anomaly_df['anomaly_type'].value_counts())
print(f"Total anomalous windows: {len(anomaly_df)}")

anomaly_df.to_csv(os.path.join(CACHE_DIR, 'injected_anomaly_labels.csv'), index=False)

# ============================================================
# 7. RAW PER-WINDOW AGGREGATION — replaces tsfresh entirely
# Simple, interpretable stats per behavior column per 15-min window:
# mean, std, min, max, median
# ============================================================
print("\nAggregating raw per-window stats (no tsfresh)...")

agg_funcs = ['mean', 'std', 'min', 'max', 'median']
behavior_agg = merged.groupby('window_id')[BEHAVIOR_COLS].agg(agg_funcs)
behavior_agg.columns = [f'{col}__{stat}' for col, stat in behavior_agg.columns]
behavior_agg = behavior_agg.fillna(0)   # std can be NaN if window has 1 row — shouldn't happen at 15min/1min data, but safe

print(f"Raw behavior feature matrix: {behavior_agg.shape[0]} windows x {behavior_agg.shape[1]} columns")

merged = merged.drop(columns=['window_id'])

# ============================================================
# 8. CONTEXT VECTOR (cyclic time + weather regime) — unchanged
# ============================================================
agg_dict = {col: 'mean' for col in weather_numeric_cols}
agg_dict.update({col: 'max' for col in weather_regime_cols})

context = merged.resample(f'{WINDOW_SIZE_MIN}min').agg(agg_dict)

context['hour']      = context.index.hour
context['dayofweek'] = context.index.dayofweek

context['hour_sin']   = np.sin(2 * np.pi * context['hour'] / 24)
context['hour_cos']   = np.cos(2 * np.pi * context['hour'] / 24)
context['dow_sin']    = np.sin(2 * np.pi * context['dayofweek'] / 7)
context['dow_cos']    = np.cos(2 * np.pi * context['dayofweek'] / 7)
context['is_weekend'] = context['dayofweek'].isin([5, 6]).astype(int)

context = context.drop(columns=['hour', 'dayofweek'])

# ============================================================
# 9. ALIGN + CONCATENATE BEHAVIOR + CONTEXT + LABELS
# ============================================================
label_lookup = anomaly_df.set_index('window_id')['anomaly_type']

combined = behavior_agg.join(context, how='inner')
combined = combined.dropna()

combined['is_anomaly'] = combined.index.isin(anomaly_df['window_id']).astype(int)
combined['anomaly_type'] = combined.index.map(label_lookup).fillna('normal')
combined.index.name = 'window_id'

print(f"\nFinal combined feature matrix: {combined.shape[0]} windows x {combined.shape[1]} columns")
print(combined['is_anomaly'].value_counts())
print(f"Anomaly rate: {combined['is_anomaly'].mean()*100:.2f}%")

# ============================================================
# 10. SAVE FINAL OUTPUT
# ============================================================
output_path = os.path.join(CACHE_DIR, 'ampds_raw_labeled_features.csv')
combined.to_csv(output_path)
print(f"Saved: {output_path}")

WHE rows: 87840, P rows: 87840, weather rows: 1464


C:\Users\revan\AppData\Local\Temp\ipykernel_27060\1399180029.py:108: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  merged[weather_numeric_cols + weather_regime_cols] = merged[weather_numeric_cols + weather_regime_cols].ffill().infer_objects(copy=False)



Injected anomaly counts by type:
anomaly_type
stuck_appliance_on                  89
stuck_appliance_off                 88
appliance_unusual_hours             78
high_usage_low_occupancy            77
heating_on_warm_day                 63
weekend_pattern_on_weekday          63
weekday_pattern_on_weekend          61
sustained_overload                  53
gradual_drift_increase              40
multiple_high_power_simultaneous    34
power_spike                         34
impossible_appliance_combo          31
sensor_glitch                       24
gradual_drift_decrease              23
Name: count, dtype: int64
Total anomalous windows: 758

Aggregating raw per-window stats (no tsfresh)...
Raw behavior feature matrix: 5856 windows x 55 columns

Final combined feature matrix: 5856 windows x 71 columns
is_anomaly
0    5098
1     758
Name: count, dtype: int64
Anomaly rate: 12.94%
Saved: C:\1.Revanth\Projects\research\pipeline_cache_raw\ampds_raw_labeled_features.csv


In [5]:
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

# ============================================================
# SWEEP CONFIG
# ============================================================
alphas = [0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
n_components_list = [10, 20, 30, 40, 50, 60]

# ============================================================
# LOAD DATA — raw (no-tsfresh) dataset, window_id is the index column
# ============================================================
df = pd.read_csv(r'pipeline_cache_raw\ampds_raw_labeled_features.csv', index_col='window_id')

# ============================================================
# STRATIFIED SPLIT — same split logic used throughout
# ============================================================
df_reset = df.reset_index(drop=True)
anomaly_df = df_reset[df_reset['is_anomaly'] == 1]
normal_df = df_reset[df_reset['is_anomaly'] == 0]

anom_train, anom_temp = train_test_split(
    anomaly_df, test_size=0.30, stratify=anomaly_df['anomaly_type'], random_state=42
)
anom_val, anom_test = train_test_split(
    anom_temp, test_size=0.50, stratify=anom_temp['anomaly_type'], random_state=42
)
norm_train, norm_temp = train_test_split(normal_df, test_size=0.30, random_state=42)
norm_val, norm_test = train_test_split(norm_temp, test_size=0.50, random_state=42)

train_df = pd.concat([anom_train, norm_train]).sample(frac=1, random_state=42)
val_df   = pd.concat([anom_val, norm_val]).sample(frac=1, random_state=42)
test_df  = pd.concat([anom_test, norm_test]).sample(frac=1, random_state=42)

drop_cols = ['is_anomaly', 'anomaly_type']
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['is_anomaly']
X_val = val_df.drop(columns=drop_cols)
y_val = val_df['is_anomaly']
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['is_anomaly']

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# ============================================================
# TRAIN ON NORMAL DATA ONLY, SCALE
# ============================================================
X_train_normal = X_train[y_train == 0]
print(f"Normal training rows: {len(X_train_normal)}")

scaler = MinMaxScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
X_test_scaled = np.clip(scaler.transform(X_test), 0, None)

# ============================================================
# SCORING FUNCTION — combined input-space + latent-space reconstruction error
# ============================================================
def rganomaly_score(nmf, X, alpha):
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)
    input_error = np.linalg.norm(X - X_recon, axis=1)

    W_reconstructed = nmf.transform(X_recon)
    latent_error = np.linalg.norm(W - W_reconstructed, axis=1)

    return alpha * input_error + (1 - alpha) * latent_error

# ============================================================
# FIT + SCORE ONE (K, alpha) CONFIG
# ============================================================
def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       alpha, max_iter=3000, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_scores = rganomaly_score(nmf, X_train_normal_scaled, alpha)
    train_err = train_scores.mean()

    val_scores = rganomaly_score(nmf, X_val_scaled, alpha)
    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>3} alpha={alpha:.1f} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "n_components": n_components, "alpha": alpha,
    }

# ============================================================
# SWEEP ALPHA x N_COMPONENTS
# ============================================================
all_results = []
all_models = {}
overall_start = time.perf_counter()

for alpha in alphas:
    print(f"\n{'='*60}\nAlpha = {alpha}\n{'='*60}")
    for n_comps in n_components_list:
        out = fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val,
                                 n_components=n_comps, alpha=alpha)
        all_results.append({
            "alpha": alpha, "n_components": n_comps,
            "converged": out["converged"], "n_iter": out["n_iter"],
            "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
            "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
            "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
            "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
        })
        all_models[(alpha, n_comps)] = out

overall_time = time.perf_counter() - overall_start
results_df = pd.DataFrame(all_results).sort_values(["pr_auc", "roc_auc"], ascending=False)

print("\n" + "="*80)
print("FULL SWEEP — Validation results (sorted by PR-AUC)")
print("="*80)
print(results_df.to_string(index=False))
print(f"\nTotal sweep runtime: {overall_time:.1f}s")

# ============================================================
# PICK BEST (alpha, K) BY VALIDATION PR-AUC
# ============================================================
best_row = results_df.iloc[0]
best_alpha = best_row["alpha"]
best_k = int(best_row["n_components"])
best_model = all_models[(best_alpha, best_k)]["model"]
best_threshold = all_models[(best_alpha, best_k)]["best_threshold"]

print(f"\nBest config: alpha={best_alpha}, n_components={best_k}")
print(f"Best validation threshold = {best_threshold:.6f}")
print(f"Converged: {all_models[(best_alpha, best_k)]['converged']} "
      f"(n_iter={all_models[(best_alpha, best_k)]['n_iter']})")

# ============================================================
# FINAL TEST EVALUATION — best config only
# ============================================================
test_scores = rganomaly_score(best_model, X_test_scaled, best_alpha)
test_pred = (test_scores >= best_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print("\n" + "="*60)
print(f"FINAL RESULT — NMF (K={best_k}, alpha={best_alpha}) on RAW (no-tsfresh) features")
print("="*60)
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")
print(f"Confusion:  TP={tp}, FP={fp}, FN={fn}, TN={tn}")

# ============================================================
# PER-ANOMALY-TYPE BREAKDOWN — best config only
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_scores = rganomaly_score(best_model, X_sub_scaled, best_alpha)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print(per_type_df.to_string(index=False))

# ============================================================
# SUMMARY ACROSS ALL (alpha, K) CONFIGS — top 10
# ============================================================
print("\n" + "="*60)
print("Top 10 configs by validation PR-AUC")
print("="*60)
print(results_df.head(10).to_string(index=False))

Train: (4098, 69) | Val: (879, 69) | Test: (879, 69)
Normal training rows: 3568

Alpha = 0.2
K= 10 alpha=0.2 |            OK | n_iter=   80 | fit_time=   0.1s | train_err=0.1005 | roc_auc=0.5762 | pr_auc=0.1550
K= 20 alpha=0.2 |            OK | n_iter= 1184 | fit_time=   1.8s | train_err=0.0363 | roc_auc=0.6212 | pr_auc=0.1907
K= 30 alpha=0.2 |            OK | n_iter= 1571 | fit_time=   4.1s | train_err=0.0150 | roc_auc=0.6293 | pr_auc=0.2260
K= 40 alpha=0.2 | NOT CONVERGED | n_iter= 3000 | fit_time=  13.9s | train_err=0.0229 | roc_auc=0.6208 | pr_auc=0.2054
K= 50 alpha=0.2 | NOT CONVERGED | n_iter= 3000 | fit_time=  21.4s | train_err=0.0696 | roc_auc=0.6095 | pr_auc=0.2331
K= 60 alpha=0.2 |            OK | n_iter= 1849 | fit_time=  18.4s | train_err=0.0659 | roc_auc=0.6005 | pr_auc=0.2050

Alpha = 0.4
K= 10 alpha=0.4 |            OK | n_iter=   80 | fit_time=   0.1s | train_err=0.2010 | roc_auc=0.5762 | pr_auc=0.1550
K= 20 alpha=0.4 |            OK | n_iter= 1184 | fit_time=   1.7s | 